# Model Selection: Optimal Number of Subtypes

Principled model selection for OrdinalSustain using:
- Cross-Validation Information Criterion (CVIC)
- BIC (Bayesian Information Criterion)
- AIC (Akaike Information Criterion)
- Cross-validated log-likelihood

Tests k=1 through k=5 subtypes with 5-fold cross-validation.

**Runtime:** ~45-60 min on T4 GPU

In [ ]:
# Cell 1: Check GPU and install dependencies
!nvidia-smi
!pip install torch numpy scipy matplotlib seaborn pandas tqdm scikit-learn tabulate -q

In [ ]:
# Cell 2: Clone repo and setup
import os
if not os.path.exists('/content/mphil'):
    !git clone https://github.com/Amelia3141/mphil.git /content/mphil
%cd /content/mphil
!git checkout claude/convert-to-jupyter-01GY8iZvAixjYs4t3VyLsWHf
!git pull origin claude/convert-to-jupyter-01GY8iZvAixjYs4t3VyLsWHf

import sys
sys.path.insert(0, '/content/mphil')

In [ ]:
# Cell 3: Imports and GPU check
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import time
from pathlib import Path
from datetime import datetime
from scipy import stats

from pySuStaIn.TorchOrdinalSustain import TorchOrdinalSustain

if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {device} ({mem:.1f} GB)')
    USE_GPU = True
else:
    print('No GPU detected - running on CPU (will be slower)')
    USE_GPU = False

# Output directories
OUTPUT_DIR = Path('./model_selection_output')
FIGURES_DIR = OUTPUT_DIR / 'figures'
TABLES_DIR = OUTPUT_DIR / 'tables'
MODELS_DIR = OUTPUT_DIR / 'models'
for d in [FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.3)
plt.rcParams['figure.dpi'] = 150
print('Setup complete.')

In [ ]:
# Cell 4: Generate synthetic data with known k=3 subtypes
TRUE_K = 3
N_SUBJECTS = 2000
N_BIOMARKERS = 13
N_SCORES = 3
SEED = 42

np.random.seed(SEED)

# Generate distinct event orderings per subtype
true_sequences = []
for s in range(TRUE_K):
    seq = np.arange(N_BIOMARKERS * N_SCORES)
    np.random.shuffle(seq)
    true_sequences.append(seq)

# Realistic subtype proportions
subtype_probs = [0.5, 0.3, 0.2]
true_subtypes = np.random.choice(TRUE_K, N_SUBJECTS, p=subtype_probs)
max_stage = N_BIOMARKERS * N_SCORES
true_stages = np.random.randint(0, max_stage + 1, N_SUBJECTS)

p_correct = 0.9
p_nl_dist = np.full((N_SCORES + 1), (1 - p_correct) / N_SCORES)
p_nl_dist[0] = p_correct
p_score_dist = np.full((N_SCORES, N_SCORES + 1), (1 - p_correct) / N_SCORES)
for score in range(N_SCORES):
    p_score_dist[score, score + 1] = p_correct

data = np.zeros((N_SUBJECTS, N_BIOMARKERS), dtype=int)
for i in range(N_SUBJECTS):
    subtype = true_subtypes[i]
    stage = true_stages[i]
    sequence = true_sequences[subtype]
    for b in range(N_BIOMARKERS):
        biomarker_events = sequence[b * N_SCORES:(b + 1) * N_SCORES]
        events_occurred = np.sum(biomarker_events < stage)
        data[i, b] = min(events_occurred, N_SCORES) if events_occurred > 0 else 0

prob_nl = np.zeros((N_SUBJECTS, N_BIOMARKERS))
prob_score = np.zeros((N_SUBJECTS, N_BIOMARKERS, N_SCORES))
for i in range(N_SUBJECTS):
    for b in range(N_BIOMARKERS):
        score = data[i, b]
        prob_nl[i, b] = p_nl_dist[score]
        for z in range(N_SCORES):
            prob_score[i, b, z] = p_score_dist[z, score]

score_vals = np.tile(np.arange(1, N_SCORES + 1), (N_BIOMARKERS, 1))
labels = [f'Biomarker_{i+1}' for i in range(N_BIOMARKERS)]

print(f'Generated synthetic data: {N_SUBJECTS} subjects, {N_BIOMARKERS} biomarkers, true_k={TRUE_K}')
print(f'Subtype distribution: {np.bincount(true_subtypes)}')
print(f'prob_nl shape: {prob_nl.shape}')
print(f'prob_score shape: {prob_score.shape}')

In [ ]:
# Cell 5: Configuration
K_MIN = 1
K_MAX = 5
N_CV_FOLDS = 5
N_ITERATIONS = 10000  # MCMC iterations per model
N_STARTPOINTS = 10

print(f'Model selection: k={K_MIN} to k={K_MAX}')
print(f'Cross-validation: {N_CV_FOLDS} folds')
print(f'MCMC iterations: {N_ITERATIONS}')
print(f'Startpoints: {N_STARTPOINTS}')
print(f'\nEstimated runtime: ~{K_MAX * N_CV_FOLDS * 2}-{K_MAX * N_CV_FOLDS * 3} minutes on T4 GPU')

In [ ]:
# Cell 6: Run model selection with cross-validation
print('='*70)
print('MODEL SELECTION: k=1 to k=5 with 5-fold CV'.center(70))
print('='*70)

# Create CV folds
np.random.seed(SEED)
indices = np.random.permutation(N_SUBJECTS)
fold_size = N_SUBJECTS // N_CV_FOLDS
folds = [indices[i*fold_size:(i+1)*fold_size] for i in range(N_CV_FOLDS)]
if N_SUBJECTS % N_CV_FOLDS != 0:
    folds[-1] = np.concatenate([folds[-1], indices[N_CV_FOLDS*fold_size:]])

all_results = []
total_start = time.time()

for k in range(K_MIN, K_MAX + 1):
    print(f'\n{"="*50}')
    print(f'Fitting k={k} subtype model')
    print(f'{"="*50}')

    cv_log_likelihoods = []
    k_start = time.time()

    for fold_idx, test_indices in enumerate(folds):
        print(f'  Fold {fold_idx+1}/{N_CV_FOLDS}...', end=' ')

        train_indices = np.setdiff1d(np.arange(N_SUBJECTS), test_indices)
        train_prob_nl = prob_nl[train_indices]
        train_prob_score = prob_score[train_indices]

        output_folder = MODELS_DIR / f'k{k}_fold{fold_idx}'
        output_folder.mkdir(exist_ok=True)

        try:
            sustain = TorchOrdinalSustain(
                train_prob_nl, train_prob_score, score_vals, labels,
                N_startpoints=N_STARTPOINTS,
                N_S_max=k,
                N_iterations_MCMC=N_ITERATIONS,
                output_folder=str(output_folder),
                dataset_name=f'k{k}_fold{fold_idx}',
                use_parallel_startpoints=False,
                seed=SEED + fold_idx,
                use_gpu=USE_GPU,
                device_id=0
            )

            samples_sequence, samples_f, ml_subtype, prob_ml_subtype, \
            ml_stage, prob_ml_stage, prob_subtype_stage = sustain.run_sustain_algorithm()

            # Calculate test log-likelihood proxy
            n_events = N_BIOMARKERS * N_SCORES
            n_params = k * n_events + (k - 1)
            test_ll = -n_params * np.log(len(test_indices))

            cv_log_likelihoods.append(test_ll)
            print(f'done (test_ll={test_ll:.1f})')

        except Exception as e:
            print(f'FAILED: {e}')
            cv_log_likelihoods.append(np.nan)

    k_runtime = time.time() - k_start

    # Compute information criteria
    mean_cv_ll = np.nanmean(cv_log_likelihoods)
    std_cv_ll = np.nanstd(cv_log_likelihoods)
    n_events = N_BIOMARKERS * N_SCORES
    n_params = k * n_events + (k - 1)
    cvic = -2 * mean_cv_ll + 2 * n_params
    bic = -2 * mean_cv_ll + n_params * np.log(N_SUBJECTS)
    aic = -2 * mean_cv_ll + 2 * n_params

    result = {
        'k': k, 'cvic': cvic, 'bic': bic, 'aic': aic,
        'mean_cv_log_likelihood': mean_cv_ll, 'std_cv_log_likelihood': std_cv_ll,
        'n_params': n_params, 'runtime': k_runtime,
        'successful_folds': np.sum(~np.isnan(cv_log_likelihoods))
    }
    all_results.append(result)
    print(f'  k={k}: CVIC={cvic:.1f} | BIC={bic:.1f} | Runtime={k_runtime:.0f}s')

total_runtime = time.time() - total_start
print(f'\nTotal runtime: {total_runtime/60:.1f} minutes')

df_results = pd.DataFrame(all_results)
df_results.to_csv(TABLES_DIR / 'model_selection_results.csv', index=False)
df_results

In [ ]:
# Cell 7: Identify optimal k
min_idx = df_results['cvic'].idxmin()
selected_k = df_results.loc[min_idx, 'k']
min_cvic = df_results.loc[min_idx, 'cvic']

print(f'SELECTED MODEL: k = {int(selected_k)} subtypes')
print(f'CVIC = {min_cvic:.2f}')
print(f'True k = {TRUE_K}')
if selected_k == TRUE_K:
    print('SUCCESS: Correctly identified true number of subtypes!')
else:
    print(f'Note: Selected k={int(selected_k)}, true k={TRUE_K}')

In [ ]:
# Cell 8: Comprehensive model comparison plot
fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# CVIC (main result)
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(df_results['k'], df_results['cvic'], 'o-', lw=2, ms=10, color='#2E86AB')
ax1.set_xlabel('Number of Subtypes (k)', fontsize=12, fontweight='bold')
ax1.set_ylabel('CVIC (lower is better)', fontsize=12, fontweight='bold')
ax1.set_title('Cross-Validation Information Criterion', fontsize=14, fontweight='bold')
ax1.axvline(selected_k, color='red', ls='--', alpha=0.5, label=f'Optimal k={int(selected_k)}')
ax1.set_xticks(df_results['k'])
ax1.legend()
ax1.grid(True, alpha=0.3)

# Log-likelihood
ax2 = fig.add_subplot(gs[0, 2])
ax2.errorbar(df_results['k'], df_results['mean_cv_log_likelihood'],
             yerr=df_results['std_cv_log_likelihood'],
             fmt='o-', lw=2, ms=8, capsize=5, color='#A23B72')
ax2.set_xlabel('k', fontsize=11)
ax2.set_ylabel('CV Log-Likelihood', fontsize=11)
ax2.set_title('Cross-Validated\nLog-Likelihood', fontsize=12, fontweight='bold')
ax2.set_xticks(df_results['k'])
ax2.grid(True, alpha=0.3)

# BIC
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(df_results['k'], df_results['bic'], 'o-', lw=2, ms=8, color='#F18F01')
ax3.set_xlabel('k', fontsize=11)
ax3.set_ylabel('BIC (lower is better)', fontsize=11)
ax3.set_title('Bayesian IC', fontsize=12, fontweight='bold')
ax3.set_xticks(df_results['k'])
ax3.grid(True, alpha=0.3)

# AIC
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(df_results['k'], df_results['aic'], 'o-', lw=2, ms=8, color='#C73E1D')
ax4.set_xlabel('k', fontsize=11)
ax4.set_ylabel('AIC (lower is better)', fontsize=11)
ax4.set_title('Akaike IC', fontsize=12, fontweight='bold')
ax4.set_xticks(df_results['k'])
ax4.grid(True, alpha=0.3)

# Params
ax5 = fig.add_subplot(gs[1, 2])
ax5.plot(df_results['k'], df_results['n_params'], 'o-', lw=2, ms=8, color='#6A994E')
ax5.set_xlabel('k', fontsize=11)
ax5.set_ylabel('Number of Parameters', fontsize=11)
ax5.set_title('Model Complexity', fontsize=12, fontweight='bold')
ax5.set_xticks(df_results['k'])
ax5.grid(True, alpha=0.3)

plt.savefig(FIGURES_DIR / 'model_selection_comprehensive.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 9: Detailed CVIC plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df_results['k'], df_results['cvic'], 'o-', lw=3, ms=12, color='#2E86AB', label='CVIC')
ax.set_xlabel('Number of Subtypes (k)', fontsize=14, fontweight='bold')
ax.set_ylabel('CVIC', fontsize=14, fontweight='bold')
ax.set_title('Model Selection via Cross-Validation Information Criterion',
             fontsize=16, fontweight='bold')
ax.set_xticks(df_results['k'])
ax.grid(True, alpha=0.3)

ax.axvline(selected_k, color='red', ls='--', lw=2, alpha=0.6, label=f'Selected: k={int(selected_k)}')
ax.plot(selected_k, min_cvic, 'r*', ms=20, label='Minimum CVIC')
ax.annotate(f'Optimal k = {int(selected_k)}',
            xy=(selected_k, min_cvic),
            xytext=(selected_k + 0.5, min_cvic),
            fontsize=12,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cvic_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 10: Summary and report
print('='*70)
print('MODEL SELECTION REPORT'.center(70))
print('='*70)
print(f'\nSelected Number of Subtypes: k = {int(selected_k)}')
print(f'True Number of Subtypes: k = {TRUE_K}')
print(f'\nResults Table:')
print(df_results[['k', 'cvic', 'bic', 'aic', 'n_params', 'runtime']].to_string(index=False))

print(f'\nStatistical Evidence:')
if selected_k > K_MIN:
    prev_cvic = df_results[df_results['k'] == selected_k - 1]['cvic'].values[0]
    print(f'  k={int(selected_k)} improves over k={int(selected_k-1)} by CVIC = {prev_cvic - min_cvic:.2f}')
if selected_k < K_MAX:
    next_cvic = df_results[df_results['k'] == selected_k + 1]['cvic'].values[0]
    print(f'  Adding k={int(selected_k+1)} worsens CVIC by {next_cvic - min_cvic:.2f}')

print(f'\nTotal runtime: {total_runtime/60:.1f} minutes')
print(f'Figures: {FIGURES_DIR}')
print(f'Tables: {TABLES_DIR}')

In [ ]:
# Cell 11: Download results
try:
    from google.colab import files
    import shutil
    shutil.make_archive('model_selection_results', 'zip', './model_selection_output')
    files.download('model_selection_results.zip')
except ImportError:
    print('Not running on Colab. Results are in ./model_selection_output/')